In [0]:
dbutils.widgets.text("PromotionName", "4P Beta")
PromotionName = dbutils.widgets.get("PromotionName")

dbutils.widgets.text("job_id","-1")
JobId = dbutils.widgets.get("job_id")

dbutils.widgets.text("run_id","-1")
PipelineRunId = dbutils.widgets.get("run_id")

ConfigID=dbutils.widgets.text("ConfigID","52")
ConfigID=dbutils.widgets.get("ConfigID")

dbutils.widgets.text('sourceTypeId','15')
sourceTypeId = dbutils.widgets.get('sourceTypeId')

dbutils.widgets.text('CreatedBy','ADB_ProcessCosumer_InvoiceAddendumDetails')
CreatedBy = dbutils.widgets.get('CreatedBy')

dbutils.widgets.text('silverpath_UlLogs','/mnt/silver/FACTULLogs')
silverpath_UlLogs = dbutils.widgets.get('silverpath_UlLogs')

dbutils.widgets.text('silverpath_InvoiceAddendumDetails','/mnt/silver/FACTInvoiceAddendumDetails')
silverpath_InvoiceAddendumDetails = dbutils.widgets.get('silverpath_InvoiceAddendumDetails')

dbutils.widgets.text('silverpath_RulesEngineAuditLog','/mnt/silver/FACTRulesEngineAuditLog')
silverpath_RulesEngineAuditLog = dbutils.widgets.get('silverpath_RulesEngineAuditLog')

dbutils.widgets.text('EmailNotificationID','2')
EmailNotificationID = dbutils.widgets.get('EmailNotificationID')

dbutils.widgets.text('UpdatedBy_CyclesOverride','CyclesOverride')
UpdatedBy_CyclesOverride = dbutils.widgets.get('UpdatedBy_CyclesOverride')

checkPointLocation = "/_checkpoints/"

In [0]:
import traceback

In [0]:
%run
/Configurations/EmailNotificationConfiguration

In [0]:
%run /Configurations/Init_Scripts

In [0]:
%run ./PopulateComments_GenericFunctions

In [0]:
%run ./Process_InvoiceException

In [0]:
%run ./Process_ConsumerSubscription_Live

In [0]:
%run ./Process_InvoiceAddendumDetails_Live

In [0]:
%run ./Process_ConsumerSubscription_TruUp

In [0]:
%run ./Process_InvoiceAddendumDetails_TruUp

In [0]:
def processConsumerShipToSoldToBasedRules(df_CycleData_AGG, UpdatedBy=None):
    
    df_FACTInvoiceAddendum = (spark.read.table('Promotion.FACT_InvoiceAddendum')
                                    .groupBy("PromotionUUID")
                                    .agg(max("InvoiceDate").alias("MaxInvoiceDate")))
    
    df_FACTConsumerSubscription_IAD = (spark.read.table('Promotion.FACT_ConsumerSubscription')
                                .filter("VersionId = 1 AND SubscriptionEndDate >= date_sub(current_date(),300)")
                                .withColumn('PilotSubscriptionFlag',coalesce(col('PilotSubscriptionFlag'),lit('N')))
                                .withColumn('lagSubscriptionEndDate',lag("SubscriptionEndDate").over(Window.partitionBy("ShipToAccountUUID", "CoolSculptingID").orderBy(asc("SubscriptionEndDate"))))
                                .withColumn('MaxSubscriptionEndDate',max("SubscriptionEndDate").over(Window.partitionBy("ShipToAccountUUID", "CoolSculptingID")))
                                .withColumn('PreviousSubscriptionEndDate',when(col('lagSubscriptionEndDate').isNull(),
                                                                                date_sub(col('SubscriptionStartDate'), 150))
                                                                            .otherwise(col('lagSubscriptionEndDate')))
                    )
    
    
    df_CycleData_Consumer = (df_CycleData_AGG.join(df_FACTConsumerSubscription_IAD,
                                         (df_CycleData_AGG.CoolSculptingID == df_FACTConsumerSubscription_IAD.CoolSculptingID) &
                                         ((df_CycleData_AGG.ShipToAccountUUID == df_FACTConsumerSubscription_IAD.ShipToAccountUUID) 
                                          |
                                          ((df_CycleData_AGG.SoldToAccountID == df_FACTConsumerSubscription_IAD.SoldToAccountID) &
                                           (df_CycleData_AGG.ShipToAccountUUID != df_FACTConsumerSubscription_IAD.ShipToAccountUUID))
                                         )&
                                         (df_CycleData_AGG.PromotionUUID == df_FACTConsumerSubscription_IAD.PromotionUUID) &
                                         (df_CycleData_AGG.CycleDate.between(df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,
                                                                             df_FACTConsumerSubscription_IAD.SubscriptionEndDate))
                                         ,
                                         "left")
                                        .select(
                                                df_CycleData_AGG.CycleDataUUID,
                                                df_CycleData_AGG.CoolSculptingID,
                                                df_CycleData_AGG.ShipToAccountUUID,
                                                df_CycleData_AGG.ShipToAccountID,
                                                df_CycleData_AGG.PromotionUUID,
                                                df_CycleData_AGG.CycleDate,
                                                df_CycleData_AGG.CycleCount,
                                                df_CycleData_AGG.MinCycleTmstmp,
                                                df_CycleData_AGG.SoldToAccountID,
                                                df_CycleData_AGG.FlagType,
                                                df_CycleData_AGG.FlagSubType,
                                                df_CycleData_AGG.SmartCardSerialNumber,
                                                df_CycleData_AGG.PlanType,
                                                df_CycleData_AGG.IsValidEfficacy,
                                                df_CycleData_AGG.ApplicatorSerialNumber,
                                                df_CycleData_AGG.CycleID,
                                                df_FACTConsumerSubscription_IAD.ConsumerSubscriptionUUID,
                                                df_FACTConsumerSubscription_IAD.SubscriptionStartDate,
                                                df_FACTConsumerSubscription_IAD.SubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.VersionID,
                                                df_FACTConsumerSubscription_IAD.Comments,
                                                df_FACTConsumerSubscription_IAD.PilotSubscriptionFlag,
                                                df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.MaxSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.InvoiceExceptionFlag
                                                )
                                        .fillna('N', subset=['InvoiceExceptionFlag'])
                                        .withColumn('CycleDataDedupFlag',row_number().over(Window.partitionBy('CycleDataUUID')
                                                                                     .orderBy(asc("PilotSubscriptionFlag"))))
                                        .filter('CycleDataDedupFlag = 1')
    )

    df_CycleData_Consumer = df_CycleData_Consumer.join(df_FACTInvoiceAddendum,["PromotionUUID"],'left')

    df_CycleData_Consumer_CycleFlag = applyRules(df_CycleData_Consumer,'CycleFlag')

    df_CycleData_Consumer_CycleFlag.persist()

    df_CycleData_ConsumerSubscription_Live = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 0 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Live = df_CycleData_Consumer_CycleFlag.filter('(AdjustSubscriptionFlag = 0 OR  PerCycleFlag = 1)')

    df_CycleData_ConsumerSubscription_Adjust = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 1 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Adjust = df_CycleData_Consumer_CycleFlag.filter('PerCycleFlag = 0')

    if(df_CycleData_ConsumerSubscription_Adjust.count()>0):
        processLateCycleSubscriptionAdjustmentsTruUp(df_CycleData_ConsumerSubscription_Adjust)
        processLateCycleSubscriptionReallignmentTruUp(df_CycleData_InvoiceaddendumDetails_Adjust)
        processInvoiceAddendumDetails_LateCycleTruUp(df_CycleData_InvoiceaddendumDetails_Adjust)

    if(df_CycleData_ConsumerSubscription_Live.count()>0):
        populateConsumerSubscription(df_CycleData_ConsumerSubscription_Live)

    if(df_CycleData_InvoiceaddendumDetails_Live.count()>0):  
        processInvoiceAddendumDetails(df_CycleData_InvoiceaddendumDetails_Live)

    populateGenericComments()

    print(df_CycleData_AGG.count())
    print(df_CycleData_Consumer_CycleFlag.count())
    df_CycleData_Consumer_CycleFlag.unpersist()    


In [0]:
def processConsumerBasedRules(df_CycleData_AGG, UpdatedBy=None):
    
    df_FACTInvoiceAddendum = (spark.read.table('Promotion.FACT_InvoiceAddendum')
                                    .groupBy("PromotionUUID")
                                    .agg(max("InvoiceDate").alias("MaxInvoiceDate")))
    
    df_FACTConsumerSubscription_IAD = (spark.read.table('Promotion.FACT_ConsumerSubscription')
                                .filter("VersionId = 1 AND SubscriptionEndDate >= date_sub(current_date(),300)")
                                .withColumn('PilotSubscriptionFlag',coalesce(col('PilotSubscriptionFlag'),lit('N')))
                                .withColumn('lagSubscriptionEndDate',lag("SubscriptionEndDate").over(Window.partitionBy("CoolSculptingID").orderBy(asc("SubscriptionEndDate"))))
                                .withColumn('MaxSubscriptionEndDate',max("SubscriptionEndDate").over(Window.partitionBy("CoolSculptingID")))
                                .withColumn('PreviousSubscriptionEndDate',when(col('lagSubscriptionEndDate').isNull(),
                                                                                date_sub(col('SubscriptionStartDate'), 150))
                                                                            .otherwise(col('lagSubscriptionEndDate')))
                    )
    
    df_CycleData_Consumer = (df_CycleData_AGG.join(df_FACTConsumerSubscription_IAD,
                                         (df_CycleData_AGG.CoolSculptingID == df_FACTConsumerSubscription_IAD.CoolSculptingID) &
                                         (df_CycleData_AGG.PromotionUUID == df_FACTConsumerSubscription_IAD.PromotionUUID) &
                                         (df_CycleData_AGG.CycleDate.between(df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,df_FACTConsumerSubscription_IAD.SubscriptionEndDate))
                                         ,
                                         "left")
                                        .select(
                                                df_CycleData_AGG.CycleDataUUID,
                                                df_CycleData_AGG.CoolSculptingID,
                                                df_CycleData_AGG.ShipToAccountUUID,
                                                df_CycleData_AGG.ShipToAccountID,
                                                df_CycleData_AGG.PromotionUUID,
                                                df_CycleData_AGG.CycleDate,
                                                df_CycleData_AGG.CycleCount,
                                                df_CycleData_AGG.MinCycleTmstmp,
                                                df_CycleData_AGG.SoldToAccountID,
                                                df_CycleData_AGG.FlagType,
                                                df_CycleData_AGG.FlagSubType,
                                                df_CycleData_AGG.SmartCardSerialNumber,
                                                df_CycleData_AGG.PlanType,
                                                df_CycleData_AGG.IsValidEfficacy,
                                                df_CycleData_AGG.ApplicatorSerialNumber,
                                                df_CycleData_AGG.CycleID,
                                                df_FACTConsumerSubscription_IAD.ConsumerSubscriptionUUID,
                                                df_FACTConsumerSubscription_IAD.SubscriptionStartDate,
                                                df_FACTConsumerSubscription_IAD.SubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.VersionID,
                                                df_FACTConsumerSubscription_IAD.Comments,
                                                df_FACTConsumerSubscription_IAD.PilotSubscriptionFlag,
                                                df_FACTConsumerSubscription_IAD.PreviousSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.MaxSubscriptionEndDate,
                                                df_FACTConsumerSubscription_IAD.InvoiceExceptionFlag
                                                )
                                        .fillna('N', subset=['InvoiceExceptionFlag'])
                                        .withColumn('CycleDataDedupFlag',row_number().over(Window.partitionBy('CycleDataUUID')
                                                                                     .orderBy(desc("SubscriptionEndDate"),asc("PilotSubscriptionFlag"))))
                                        .filter('CycleDataDedupFlag = 1')
                            )

    df_CycleData_Consumer = df_CycleData_Consumer.join(df_FACTInvoiceAddendum,["PromotionUUID"],'left')

    df_CycleData_Consumer_CycleFlag = applyRules(df_CycleData_Consumer,'CycleFlag')

    df_CycleData_Consumer_CycleFlag.persist()

    df_CycleData_ConsumerSubscription_Live = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 0 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Live = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 0 OR  PerCycleFlag = 1')
    df_CycleData_ConsumerSubscription_Adjust = df_CycleData_Consumer_CycleFlag.filter('AdjustSubscriptionFlag = 1 AND PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Adjust = df_CycleData_Consumer_CycleFlag.filter('PerCycleFlag = 0')
    df_CycleData_InvoiceaddendumDetails_Live = df_CycleData_Consumer_CycleFlag.filter('(AdjustSubscriptionFlag = 0 OR  PerCycleFlag = 1)')

    if(df_CycleData_ConsumerSubscription_Adjust.count()>0):
        print("Running for Tru up logic")        
        processLateCycleSubscriptionAdjustmentsTruUp_ConsumerBased(df_CycleData_ConsumerSubscription_Adjust)
        processLateCycleSubscriptionReallignmentTruUp_ConsumerBased(df_CycleData_InvoiceaddendumDetails_Adjust)
        processInvoiceAddendumDetails_LateCycleTruUp_ConsumerBased(df_CycleData_InvoiceaddendumDetails_Adjust)

    if(df_CycleData_ConsumerSubscription_Live.count()>0):
        print("Running for ConsumerSubscription")        
        populateConsumerSubscription_ConsumerBased(df_CycleData_ConsumerSubscription_Live)

    if(df_CycleData_InvoiceaddendumDetails_Live.count()>0):  
        print("Running for InvoiceAddendumDetails_Live")  
        processInvoiceAddendumDetails_ConsumerBased(df_CycleData_InvoiceaddendumDetails_Live)

    # populateGenericComments_ConsumerBased()
    populateGenericComments()

    print(df_CycleData_AGG.count())
    print(df_CycleData_Consumer_CycleFlag.count())
    df_CycleData_Consumer_CycleFlag.unpersist()    


In [0]:
def processCycleData(df_CycleData, UpdatedBy=None):

    if(UpdatedBy == UpdatedBy_CyclesOverride):
        df_ValidCycleData = getValidPromotionCycles_Overrides(df_CycleData)
    else:
        df_ValidCycleData = getValidPromotionCycles(df_CycleData)
    
    df_AccountFlags =  getAccountFlags(df_ValidCycleData)
    df_CycleData_AGG = df_ValidCycleData.join(df_AccountFlags,['ShipToAccountUUID','PromotionUUID','CycleDate'],'left') 

    df_CycleData_Consumer = df_CycleData_AGG.filter('PromotionRuleType = "Consumer"')
    df_CycleData_ConsumerShipToSoldTo = df_CycleData_AGG.filter('PromotionRuleType = "Consumer-ShipTo-SoldTo"')    

    print('processConsumerBasedRules')
    processConsumerBasedRules(df_CycleData_Consumer, UpdatedBy=UpdatedBy)
    print('processConsumerShipToSoldToBasedRules')    
    processConsumerShipToSoldToBasedRules(df_CycleData_ConsumerShipToSoldTo, UpdatedBy=UpdatedBy)
    


In [0]:
def upsertToDelta(microBatchOutputDF, batchId):     
    log_Stream = {
        "ConfigID" : ConfigID,
        "SourceTypeID" : sourceTypeId,
        "Source" : "silverzone.factullogs",
        "Destination" : "promotion.fact_invoiceaddendumdetails",
        "Run_ID": str(batchId),
        "Job_ID": str(JobId)
        }
    logdf_stream = spark.createDataFrame([log_Stream])

    log_Exception = {
        "ConfigID" : ConfigID,
        "SourceTypeID" : sourceTypeId,
        "Source" : "Promotion.FACT_InvoiceException",
        "Destination" : "promotion.fact_invoiceaddendumdetails",
        "Run_ID": str(batchId),
        "Job_ID": str(JobId)
        }
    logdf_Exception = spark.createDataFrame([log_Exception])

    log_CyclesOverride = {
        "ConfigID" : ConfigID,
        "SourceTypeID" : sourceTypeId,
        "Source" : "Promotion.DIM_CyclesOverride",
        "Destination" : "promotion.fact_invoiceaddendumdetails",
        "Run_ID": str(batchId),
        "Job_ID": str(JobId)
        }
    logdf_CyclesOverride = spark.createDataFrame([log_CyclesOverride])
    batchEnd(q,batchId)
    print("Running for BatchID:"+str(batchId))
    
    DF_InvoiceException = (spark.read.table('Promotion.FACT_InvoiceException').filter('IsProcessedFlag is NULL'))
    if(DF_InvoiceException.count()>0):
        try:
            print("Process Invoice Exception:"+str(DF_InvoiceException.count()))
            processInvoiceException()
            logIntoStreamLogTable(logdf_Exception,CreatedBy,"Succeeded")
        except Exception as e:
            ExceptionTraceback = traceback.format_exc()
            ErrorMessage = ExceptionTraceback + str(e) 
            print(ExceptionTraceback)
            logIntoStreamLogTable(logdf_Exception,CreatedBy,"Failed",DF_InvoiceException,ErrorMessage)
            streamEmailNotification(EmailNotificationID,DF_InvoiceException)

    df_DIMCyclesOverride_input = spark.read.table('promotion.DIM_CyclesOverride').filter("coalesce(WorkflowStatus,'') = ''")
    df_DIMCyclesOverride = df_DIMCyclesOverride_input.dropDuplicates(['CycleId','PromotionUUID','CreatedDate'])
    if(df_DIMCyclesOverride.count()>0):
        try:
            try:
                print("Process Cycles Override:"+str(df_DIMCyclesOverride.count()))            
                setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride,'InProgress',UpdatedBy_CyclesOverride)
                df_DIMCyclesOverride_InProgress_input = spark.read.table('promotion.DIM_CyclesOverride').filter('WorkflowStatus = "InProgress"')
                df_DIMCyclesOverride_InProgress = df_DIMCyclesOverride_InProgress_input.dropDuplicates(['CycleId','PromotionUUID','CreatedDate'])
                try:
                    processCycleData(df_DIMCyclesOverride_InProgress,UpdatedBy_CyclesOverride)
                    setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride_InProgress,'Success',UpdatedBy_CyclesOverride)
                    logIntoStreamLogTable(logdf_CyclesOverride,CreatedBy,"Succeeded")                
                except Exception as e:
                    setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride_InProgress,'Fail',UpdatedBy_CyclesOverride)
                    ExceptionTraceback = traceback.format_exc()
                    ErrorMessage = ExceptionTraceback + str(e) 
                    print(ExceptionTraceback)
                    logIntoStreamLogTable(logdf_CyclesOverride,CreatedBy,"Failed",df_DIMCyclesOverride_InProgress,ErrorMessage)
                    streamEmailNotification(EmailNotificationID,df_DIMCyclesOverride_InProgress)
            except Exception as e:
                setWorkFlowStatus_CyclesOverride(df_DIMCyclesOverride,'Fail',UpdatedBy_CyclesOverride)
                ExceptionTraceback = traceback.format_exc()
                ErrorMessage = ExceptionTraceback + str(e) 
                print(ExceptionTraceback)                
                logIntoStreamLogTable(logdf_CyclesOverride,CreatedBy,"Failed",df_DIMCyclesOverride,ErrorMessage)
                streamEmailNotification(EmailNotificationID,df_DIMCyclesOverride)
        except Exception as e:
            ExceptionTraceback = traceback.format_exc()
            ErrorMessage = ExceptionTraceback + str(e) 
            print(ExceptionTraceback)
            logIntoStreamLogTable(logdf_CyclesOverride,CreatedBy,"Failed",df_DIMCyclesOverride,ErrorMessage)
            streamEmailNotification(EmailNotificationID,df_DIMCyclesOverride)
    try:
        print("Process P3 Cycles")
        processCycleData(microBatchOutputDF)
        logIntoStreamLogTable(logdf_stream,CreatedBy,"Succeeded")
    except Exception as e:
        ExceptionTraceback = traceback.format_exc()
        ErrorMessage = ExceptionTraceback + str(e) 
        print(ExceptionTraceback)
        logIntoStreamLogTable(logdf_stream,CreatedBy,"Failed",microBatchOutputDF,ErrorMessage)
        streamEmailNotification(EmailNotificationID,microBatchOutputDF)

In [0]:
df_Source = (spark.readStream
                .option("cloudFiles.maxFilesPerTrigger", 5000)
                .option("cloudFiles.maxBytesPerTrigger", '10g')
                .option("skipChangeCommits", "true")
                .format("delta")
                .load(silverpath_UlLogs)
                .withColumnRenamed('CardSPSerialNbr', 'SmartCardSerialNumber')
                .withColumnRenamed('HdrStartTimeGeneratedTmstmp', 'BeginTimeStamp')
                .withColumnRenamed('EventEndTmstmp', 'EndTimeStamp')
                .withColumn('PlanType', coalesce(col('PlanType'), lit('ComprehensiveTreatmentPlan')))
                .withColumn('ApplicatorConfigurantionNbr', left(col('ApplicatorConfigurantionNbr'), lit(7))) 
                .select('CycleID', 'ShipToAccountID', 'SoldToAccountID', 'HdrStartDateGeneratedDt', 'BeginTimeStamp', 'EndTimeStamp','SmartCardSerialNumber', 'CoolSculptingID', 'FileNameDeviceTypeCd', 'PlanType','ApplicatorConfigurantionNbr','ApplicatorSerialNbr')
             )

In [0]:
q=(df_Source.writeStream
                  .format("delta")
                  .queryName("Process_PromotionULCycles_Streaming")
                  .trigger(processingTime='10 seconds')
                  .foreachBatch(upsertToDelta)
                  .option("checkpointLocation", silverpath_InvoiceAddendumDetails+checkPointLocation)
                  .outputMode("update")
                  .start()
)

In [0]:
stop_process = 1
stop_process =  graceStop(q,1)

In [0]:
q.awaitTermination() 